# Module 02 — DQN on LunarLander

Replace the table with a neural network and land the lunar module. See [the lesson](README.md).

> Tip: in Colab, set **Runtime → Change runtime type → GPU**.

In [ ]:
# === Colab setup: run me first ===
import os, sys
if not os.path.exists('rl'):
    # On Colab, clone the repo so the `rl` package is importable.
    !git clone https://github.com/anhduckkzz/lunarlander.git repo && (cp -r repo/* . 2>/dev/null || true)
    !pip -q install 'gymnasium[box2d]>=0.29' torch numpy matplotlib imageio tqdm
import torch
print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## Train DQN (Double DQN is on by default)

In [ ]:
from rl.envs import make_env, env_dims
from rl.agents.dqn import DQN, DQNConfig
from rl.train import train_offpolicy
from rl.utils import ExponentialSchedule, Logger, plot_scores, set_seed

set_seed(0)
env = make_env('LunarLander-v3', seed=0)
s_dim, a_dim, _ = env_dims(env)
agent = DQN(s_dim, a_dim, DQNConfig(double=True), device=DEVICE)
eps = ExponentialSchedule(1.0, 0.01, 0.9995)
scores = train_offpolicy(agent, env, n_steps=200_000, eps_schedule=eps,
                         logger=Logger('runs/dqn_nb'), solved_at=200)
plot_scores(scores, title='DQN on LunarLander', show=True)

## Save & watch it play (records a GIF you can display)

In [ ]:
agent.save('model/dqn_nb.pth')
import imageio, numpy as np
from rl.envs import make_env
eval_env = make_env('LunarLander-v3', render_mode='rgb_array', seed=1)
frames, state, done = [], eval_env.reset()[0], False
while not done:
    frames.append(eval_env.render())
    state, r, term, trunc, _ = eval_env.step(agent.act(state, eps=0.0))
    done = term or trunc
imageio.mimsave('doc/dqn_nb.gif', frames, fps=30)
print('saved doc/dqn_nb.gif —', len(frames), 'frames')

**Ablation:** rerun with `DQNConfig(double=False)` and with `tau=1.0` (no target network). Compare the curves.